# Урок 08 - Многoагентен дизайн шаблон


## Настройка


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Защо мултиагентни системи?

Задачи от реалния свят като планиране на пътуване включват много различни видове експертиза — логистика, местни познания, бюджетиране и още. Един единствен агент, който се опитва да се справи с всичко, бързо става непрактичен.

Мултиагентните системи решават това чрез **специализация**: всеки агент се фокусира върху една област на експертиза, като произвежда по-висококачествени резултати от един генералист. Те също така подобряват **масштабируемостта** — може да добавяте нови агенти (например специалист по полети, критик на ресторанти) без да преправяте съществуващия работен поток. Агентите функционират заедно чрез структурирана линия, предавайки контекст от един към друг.


## Създаване на специализирани агенти


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Създаване на последователен работен процес

`WorkflowBuilder` ви позволява да свържете агенти в насочен граф. Тук създаваме проста двустъпкова линия: **TravelPlanner** съставя маршрута, след това **TravelConcierge** го преглежда и подобрява.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Добавяне на още агенти към работния процес

Едно от най-големите предимства на многоагентния модел е колко лесно се разширява. По-долу добавяме агент **BudgetReviewer**, който проверява плана спрямо бюджета на пътешественика, маркира елементи, които могат да надвишат лимита, и предлага алтернативи за спестяване на пари. Работният процес вече изпълнява три агента последователно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Резюме

В този урок научихте как да:

1. **Създавате специализирани агенти** — всеки с фокусирана роля (планиране, рецепция, преглед на бюджета).
2. **Свързвате агентите в последователен работен процес** с помощта на `WorkflowBuilder` и `add_edge`.
3. **Поточно извеждате** от многосрочна тръба, следейки кой агент говори.
4. **Разширявате работния процес** чрез добавяне на нови агенти към веригата без да модифицирате съществуващите.

Моделът с много агенти поддържа всеки агент прост, като същевременно произвежда по-богати и по-задълбочено прегледани резултати, отколкото който и да било агент би могъл да постигне сам.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
